# Bước 1 — Chọn dữ liệu và xác định bài toán

**Đề tài:** Bệnh viện — dự đoán tái nhập viện trong vòng 30 ngày.

**Câu hỏi nghiệp vụ:** “Sau khi xuất viện, nên gọi theo dõi cho bệnh nhân nào trước?”

- Một bản ghi = một lượt nhập viện, không nhất thiết là một bệnh nhân duy nhất.
- Biến đích `readmitted_30d`: 1 nếu `readmitted == "<30"`, ngược lại là 0.
- Chỉ khảo sát và xác định dữ liệu trong bước 1; chưa làm sạch hoặc huấn luyện.
- Nguồn: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008
- Giấy phép theo UCI: CC BY 4.0 (ghi công nguồn).

In [ ]:
from pathlib import Path
import pandas as pd

# Notebook được đặt trong notebooks/ của thư mục dự án.
# Khởi chạy JupyterLab ở thư mục gốc Readmission_Project.
root = Path.cwd()
if not (root / "data" / "raw").exists():
    root = root.parent
raw_dir = root / "data" / "raw"
file_path = raw_dir / "diabetic_data.csv"
assert file_path.is_file(), f"Chưa tìm thấy {file_path}. Hãy tải và giải nén dữ liệu từ UCI."
print("Đường dẫn dữ liệu:", file_path.resolve())

In [ ]:
df = pd.read_csv(file_path, low_memory=False)
print("Số dòng, số cột:", df.shape)
print("Tên các cột:", df.columns.tolist())
display(df.head())

In [ ]:
print("Kiểu dữ liệu:")
print(df.dtypes)

print("\nThông tin tổng quan:")
df.info()

In [ ]:
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)
print("Phân bố biến mục tiêu nhị phân:")
print(df["readmitted_30d"].value_counts().rename(index={0: "Không dưới 30 ngày", 1: "Dưới 30 ngày"}))
print("Tỉ lệ tái nhập viện dưới 30 ngày: {:.2%}".format(df["readmitted_30d"].mean()))

In [ ]:
print("Số lượt nhập viện:", len(df))
print("Số bệnh nhân duy nhất:", df["patient_nbr"].nunique())
print("Số lượt trùng encounter_id:", df["encounter_id"].duplicated().sum())
print("Số dòng có dấu hỏi ? theo cột (chưa xử lý):")
question_counts = df.astype("string").eq("?").sum().sort_values(ascending=False)
display(question_counts[question_counts > 0].to_frame("so_dau_hoi"))

## Ghi chú cho các bước tiếp theo

1. **Chưa thay `?` bằng NaN ở bước 1**; chỉ ghi nhận để phân tích chất lượng dữ liệu ở bước 2.
2. Cần tách train/validation/test **theo `patient_nbr`**, tránh cùng bệnh nhân xuất hiện ở nhiều tập.
3. Không sử dụng cột `readmitted` làm đặc trưng đầu vào để dự đoán `readmitted_30d`.
4. Bước 1 hoàn thành khi đọc được CSV, thống kê số dòng/cột và phân bố nhãn, lưu bằng chứng dữ liệu tải thành công.
5. Đồ án chạy nội bộ; không gửi dữ liệu bệnh nhân lên bên thứ ba.

In [ ]:
from pathlib import Path

output_path = Path("../data/processed/step1_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

print("Đã lưu dữ liệu:", output_path)
print("Kích thước:", df.shape)